# Refusal calibration — whole pipeline, one notebook

Probe -> label -> build -> train -> generate -> score, top to bottom. No
restarts, no download/re-upload between stages.

**Setup:** New Notebook -> Accelerator **GPU T4 x2**, Internet **on**,
Add Data -> your uploaded repo dataset (`refusal-calibration`).

**Every stage is resumable.** The probe skips done questions, training skips
runs that already have an adapter, generation skips arms with a responses file.
Rerun after a dead session and it picks up where it stopped — "Run All" is
always the right button.

**This does not fit in one 9h session, by design.** Full scale is ~14 training
runs plus ~16 eval arms. Kaggle gives 30h/week, so plan on three:

| session | `CFG["stages"]` | ~time |
|---|---|---|
| 1 | `probe, build, train` with `runs[:5]` | ~8h |
| 2 | `train` with the rest | ~8h |
| 3 | `generate, score` | ~4h |

Set `CFG["stages"]` to what you want this session to do. Commit the notebook
(Save Version -> Save & Run All) so `/kaggle/working` persists between them, or
download `results.zip` and re-upload — both work, the resume logic is
file-based.

In [ ]:
!pip install -q unsloth trl peft datasets transformers accelerate pyyaml

In [ ]:
CFG = {
    "repo_src": "/kaggle/input/refusal-calibration",
    "repo": "/kaggle/working/repo",

    "stages": ["probe", "build", "train", "generate", "score"],   # trim per session

    # stage 1 — probe. k=16 halves the resolution error on p_correct vs k=8,
    # which is what decides answerable/borderline/unknown for every item.
    "n_questions": 25000,
    "k_probe": 16,
    "temperature": 0.7,
    "probe_batch_size": 64,     # halve on CUDA OOM
    "max_new_tokens": 64,

    # stage 2 — train. Priority order; see RUNBOOK.md for what each buys.
    "runs": ["v3_mix50", "v2_mix25", "v4_mix75", "v1_mix10", "v5_mix90",
             "v12_seed1", "v13_seed2", "v6_1epoch", "v7_3epoch",
             "v8_rank8", "v9_rank32", "v10_lr1e4", "v11_bare_reason",
             "v14_qwen3b"],

    # stage 3 — eval generation. k_eval < k_probe on purpose: the confidence
    # signal only needs to rank buckets, and this cost is paid per *arm*.
    "k_eval": 8,
    "eval_batch_size": 32,
}
PROMPT_BASELINE = ('Answer the question. If you are not sure, say "I don\'t know" '
                   'and briefly say why. Never guess.')

In [ ]:
import os, shutil, subprocess, sys, glob, json, time
from collections import Counter

if not os.path.exists(CFG["repo"]):
    shutil.copytree(CFG["repo_src"], CFG["repo"])
os.chdir(CFG["repo"])
sys.path[:0] = [CFG["repo"], os.path.join(CFG["repo"], "data")]

import torch, yaml
print(torch.cuda.get_device_name(0), "| stages:", CFG["stages"])
subprocess.run([sys.executable, "tests.py"], check=True)   # logic intact before anything expensive

## Stage 1 — probe: measure what the base model actually knows

The 1.5B base is sampled k times per question; the answers *it* gets right
become the answerable class and the ones it never gets become the abstain
class. Skipped if `data/eval.jsonl` already exists.

In [ ]:
PROBE_OUT = "data/probe_raw.jsonl"
PROBE_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
NEED_PROBE = "probe" in CFG["stages"] and not os.path.exists("data/eval.jsonl")

if NEED_PROBE:
    from refusal import SYSTEM_PROMPT
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(PROBE_BASE)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    probe_model = AutoModelForCausalLM.from_pretrained(
        PROBE_BASE, torch_dtype=torch.float16, device_map="cuda")
    probe_model.eval()

    rows = [json.loads(l) for l in open("data/questions.jsonl", encoding="utf-8")][:CFG["n_questions"]]
    done = set()
    if os.path.exists(PROBE_OUT):
        done = {json.loads(l)["question"] for l in open(PROBE_OUT, encoding="utf-8")}
    rows = [r for r in rows if r["question"] not in done]
    print(len(rows), "to probe;", len(done), "already done")
else:
    rows = []
    print("skipping probe")

In [ ]:
if NEED_PROBE and rows:
    def prompt_for(q):
        return tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
            tokenize=False, add_generation_prompt=True)

    out = open(PROBE_OUT, "a", encoding="utf-8")
    bs, t0 = CFG["probe_batch_size"], time.time()
    with torch.no_grad():
        for start in range(0, len(rows), bs):
            batch = rows[start:start + bs]
            enc = tok([prompt_for(r["question"]) for r in batch], return_tensors="pt", padding=True).to("cuda")
            gen = probe_model.generate(**enc, do_sample=True, temperature=CFG["temperature"], top_p=0.95,
                                       num_return_sequences=CFG["k_probe"], max_new_tokens=CFG["max_new_tokens"],
                                       pad_token_id=tok.pad_token_id)
            texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, r in enumerate(batch):
                out.write(json.dumps({**r, "samples": texts[i * CFG["k_probe"]:(i + 1) * CFG["k_probe"]]}) + "\n")
            out.flush()
            if (start // bs) % 20 == 0:
                elapsed = (time.time() - t0) / 60
                rate = (start + len(batch)) / max(elapsed, 0.1)
                print(f"{start + len(batch)}/{len(rows)}  {elapsed:.0f} min  eta {(len(rows)-start)/rate:.0f} min", flush=True)
    out.close()

if NEED_PROBE:
    del probe_model
    torch.cuda.empty_cache()
    # eyeball one raw sample before trusting the labels — if this is prose, the
    # chat template isn't being applied and every label downstream is wrong
    print(repr(json.loads(open(PROBE_OUT, encoding="utf-8").readline())["samples"][0]))

## Stage 1b — label, freeze the eval, build the mixes (CPU, seconds)

`preflight.py` is the gate: it fails loudly here rather than three hours into
training.

In [ ]:
if "build" in CFG["stages"] and not os.path.exists("data/eval.jsonl"):
    subprocess.run([sys.executable, "-m", "data.probe"], check=True)
    subprocess.run([sys.executable, "-m", "data.build"], check=True)
subprocess.run([sys.executable, "preflight.py"], check=True)

## Stage 2 — train

One script, one config per run, only the variable named in the config differs.
A failed run prints and the loop continues — one bad config shouldn't cost the
session (RUNBOOK.md has the failure table).

In [ ]:
if "train" in CFG["stages"]:
    for name in CFG["runs"]:
        if os.path.exists(f"runs/{name}/adapter/adapter_config.json"):
            print("skip (done):", name); continue
        print("=" * 60, name, flush=True)
        t0 = time.time()
        r = subprocess.run([sys.executable, "train.py", "--config", f"configs/{name}.yaml"])
        print(f"{'done' if r.returncode == 0 else 'FAILED'} {name} in {(time.time()-t0)/60:.1f} min", flush=True)

## Stage 3 — generate responses for every arm

Arms are built from the configs, so a run trained on a different base model
automatically gets **its own** `base` and `prompt` baselines — over-refusal is
only meaningful against the same family's base, and `meta.json` records which
one each arm is scored against.

In [ ]:
from refusal import SYSTEM_PROMPT, full_precision
from probe import consistency            # same confidence definition as the labeling stage
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

items = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")]
questions = [i["question"] for i in items]
print(len(items), "eval items", Counter(i["klass"] for i in items))

# arm table: name -> (inference base, adapter dir or None, system prompt, base arm)
arms, families = {}, {}
for adapter in sorted(glob.glob("runs/*/adapter")):
    name = os.path.basename(os.path.dirname(adapter))
    base_id = full_precision(yaml.safe_load(open(f"configs/{name}.yaml"))["base_model"])
    families.setdefault(base_id, "base" if "1.5B" in base_id else f"base_{base_id.split('-')[-2].lower()}")
    arms[name] = (base_id, adapter, SYSTEM_PROMPT, families[base_id])
families.setdefault("Qwen/Qwen2.5-1.5B-Instruct", "base")   # baselines even with no adapters yet
for base_id, base_arm in families.items():
    arms[base_arm] = (base_id, None, SYSTEM_PROMPT, base_arm)
    arms[base_arm.replace("base", "prompt")] = (base_id, None, PROMPT_BASELINE, base_arm)
for n, (b, a, s, f) in arms.items():
    print(f"  {n:16s} base={b:28s} adapter={'yes' if a else '-':3s} scored_vs={f}")

In [ ]:
def build_model(base_id, adapter=None):
    tk = AutoTokenizer.from_pretrained(adapter or base_id)
    tk.padding_side = "left"
    if tk.pad_token is None:
        tk.pad_token = tk.eos_token
    m = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float16, device_map="cuda")
    if adapter:
        m = PeftModel.from_pretrained(m, adapter)
    m.eval()
    return m, tk

def generate(model, tk, system, sample, k=1):
    texts = []
    with torch.no_grad():
        for start in range(0, len(questions), CFG["eval_batch_size"]):
            batch = questions[start:start + CFG["eval_batch_size"]]
            prompts = [tk.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": q}],
                tokenize=False, add_generation_prompt=True) for q in batch]
            enc = tk(prompts, return_tensors="pt", padding=True).to("cuda")
            kw = dict(max_new_tokens=CFG["max_new_tokens"], pad_token_id=tk.pad_token_id)
            kw.update(dict(do_sample=True, temperature=CFG["temperature"], top_p=0.95,
                           num_return_sequences=k) if sample else dict(do_sample=False))
            gen = model.generate(**enc, **kw)
            texts += tk.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return texts

def run_arm(name, base_id, adapter, system, base_arm):
    path = f"runs/{name}/responses.jsonl"
    if os.path.exists(path):
        print("skip (done):", name); return
    os.makedirs(f"runs/{name}", exist_ok=True)
    t0 = time.time()
    model, tk = build_model(base_id, adapter)
    greedy = generate(model, tk, system, sample=False)
    sampled = generate(model, tk, system, sample=True, k=CFG["k_eval"])
    with open(path, "w", encoding="utf-8") as f:
        for i, q in enumerate(questions):
            f.write(json.dumps({"question": q, "response": greedy[i],
                                "confidence": round(consistency(sampled[i * CFG["k_eval"]:(i + 1) * CFG["k_eval"]]), 3)}) + "\n")
    json.dump({"base_arm": base_arm, "base_model": base_id, "adapter": adapter},
              open(f"runs/{name}/meta.json", "w"))
    del model
    torch.cuda.empty_cache()
    print(f"wrote {path}  ({(time.time()-t0)/60:.1f} min)", flush=True)

if "generate" in CFG["stages"]:
    for name in sorted(arms, key=lambda n: (arms[n][1] is not None, n)):   # baselines first
        run_arm(name, *arms[name])

## Stage 4 — score, in the same session

CPU work, seconds. Same `metrics.py` for every arm, so the numbers are
comparable by construction. Read `v12_seed1` / `v13_seed2` against `v3_mix50`
first: that spread is your noise floor, and any gap smaller than it is not a
finding no matter how good it looks on the curve.

In [ ]:
from eval import load_eval, load_records
from metrics import METRICS, ci, paired_delta, reliability, summarize
from curve import base_arm_of

frozen = load_eval()
runs = {os.path.basename(os.path.dirname(p)): os.path.dirname(p)
        for p in sorted(glob.glob("runs/*/responses.jsonl"))}
records = {n: load_records(f"{d}/responses.jsonl", frozen, base_arm_of(d)) for n, d in runs.items()}

for name, recs in records.items():
    print(summarize(recs, name)); print()

In [ ]:
# paired bootstrap vs the matching family's base — a delta without a * is not a result
for name, recs in records.items():
    base_name = json.load(open(f"{runs[name]}/meta.json"))["base_arm"] if os.path.exists(f"{runs[name]}/meta.json") else "base"
    if name == base_name or base_name not in records:
        continue
    by_q = {r["question"]: r for r in records[base_name]}
    paired = [(by_q[r["question"]], r) for r in recs if r["question"] in by_q]
    a, b = [p[0] for p in paired], [p[1] for p in paired]
    print(f"== {name} - {base_name}  (n={len(paired)})")
    for label, metric in METRICS.items():
        d, lo, hi = paired_delta(a, b, metric)
        print(f"  Δ {label:22s} {d:+6.1%}  [{lo:+.1%}, {hi:+.1%}]{'' if lo <= 0 <= hi else '  *'}")
    print()

In [ ]:
# noise floor: same recipe, three seeds. Any curve gap smaller than this spread
# is unfalsifiable — say so in the write-up rather than ranking the mixes by it.
seeds = [n for n in ("v3_mix50", "v12_seed1", "v13_seed2") if n in records]
if len(seeds) > 1:
    for label, metric in METRICS.items():
        vals = [metric(records[n])[0] for n in seeds]
        vals = [v for v in vals if v is not None]
        if vals:
            print(f"  {label:22s} " + "  ".join(f"{v:.1%}" for v in vals) +
                  f"   spread={max(vals)-min(vals):.1%}")
else:
    print("train v12_seed1 and v13_seed2 to get a noise floor")

In [ ]:
subprocess.run([sys.executable, "curve.py"], check=True)          # the deliverable
print()
print("reliability (base):")
for mid, acc, n in reliability(records["base"]):
    print(f"  conf~{mid:.2f}  acc={acc:.1%}  n={n}")

## Stage 5 — take it home

`results.zip` is small and is what the write-up is built from: the frozen eval,
its lock, the labeled corpus, the mixes and every arm's raw generations.
`adapters.zip` is ~75MB per run (more for 3B) — grab it for the versions you
intend to publish.

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip data/eval.jsonl data/eval.lock \
    data/labeled.jsonl data/mix_*.jsonl runs/*/responses.jsonl runs/*/meta.json
!cd /kaggle/working/repo && zip -qr /kaggle/working/probe.zip data/probe_raw.jsonl
!cd /kaggle/working/repo && zip -qr /kaggle/working/adapters.zip runs/*/adapter
!ls -lh /kaggle/working/*.zip

Unzip `results.zip` over the local repo and everything re-scores on your laptop
with no GPU:

```
python eval.py --run runs/v3_mix50/responses.jsonl --base runs/base/responses.jsonl --name v3_mix50 --compare
python curve.py
```

Keep `probe.zip` too — it's the raw evidence behind every label, and rebuilding
it costs a GPU session.

Something broke? `python tests.py` (is the logic intact) and `python preflight.py`
(is the data intact, which stage am I at), then the troubleshooting tables in
RUNBOOK.md.